In [ ]:
!pip install librosa

In [ ]:
import os,sys, yt_dlp, re

In [ ]:
import logging
logger = logging.getLogger(__name__)

In [ ]:
# Function to extract video ID
def extract_video_id(url: str) -> str:
    try:
        regex = r"v=([a-zA-Z0-9_-]+)"
        match = re.search(regex, url)
        logger.info(f"match: {match}")
        logger.info(f"url: {url}")
        if match:
            return match.group(1)
        logger.warning(f"No video ID found in URL: {url}")
        return ""
    except Exception as e:
        logger.error(f"Error extracting video ID from {url}: {str(e)}")
        return ""


In [ ]:
# # lf

# <!-- jr & lf -->
# https://www.youtube.com/watch?v=tlOyZSAZh2k

# <!-- em & lf -->
# https://www.youtube.com/watch?v=smK9dgdTl40

# <!-- lf & audience q & a -->
# https://www.youtube.com/watch?v=_ySbzVXiwzQ

# <!-- lf monologue -->
# https://www.youtube.com/watch?v=0m3hGZvD-0s

# # SZ
# <!-- sz -->
# https://www.youtube.com/watch?v=CkUcCcRq_eM

# <!-- sz reference audio -->
# https://www.youtube.com/watch?v=gIF_D6iUusU&t=10s


# # jh

# <!-- jh reference audio -->
# https://www.youtube.com/watch?v=smK9dgdTl40

# # TS

# <!-- ts interview -->
# https://www.youtube.com/watch?v=XnbCSboujF4

# # GA

# <!-- ga interview -->
# https://www.youtube.com/watch?v=qooTyQKxgEU
# https://www.youtube.com/watch?v=xRGZa9oyXfM

# <!-- ga reference audio -->
# https://www.youtube.com/watch?v=NBih6bQmntc

# # DG 

# <!-- dg reference audio -->
# https://www.youtube.com/watch?v=mJMwKQSovjo

# <!-- interview dg -->
# https://www.youtube.com/watch?v=YU186kXJRlo

# # AH

# <!-- interview dg & ah -->
# https://www.youtube.com/watch?v=nDLb8_wgX50

# <!-- ah monologue -->
# https://www.youtube.com/watch?v=QmOF0crdyRU


In [ ]:
video_url = "https://www.youtube.com/watch?v=0m3hGZvD-0s"

In [ ]:
video_id = extract_video_id(video_url)

In [ ]:
# Download audio
try: 
    os.makedirs('audio', exist_ok=True)
    output_dir = f'audio/{video_id}_audio.%(ext)s'
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'audio/{video_id}_audio.%(ext)s',

        'postprocessors': [{ 

            'key': 'FFmpegExtractAudio', 

            'preferredcodec': 'mp3', 

            'preferredquality': '192', 

        }], 

    } 

    with yt_dlp.YoutubeDL(ydl_opts) as ydl: 

        ydl.download([video_url]) 

    audio_path = f'audio/{video_id}_audio.mp3' 

except Exception as e: 

    logger.error(f"Error downloading audio for {video_url}: {str(e)}") 

In [ ]:
output_dir = f'audio/{video_id}_audio.mp3'

In [ ]:
# comparison of two clips:
import librosa
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
def compute_mfcc_vec(audio_file):
    y, sr = librosa.load(audio_file)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    # take the average
    mfcc_vec = np.mean(mfcc.T, axis=0).reshape(1, -1)
    return mfcc_vec


In [ ]:

def compare_mfcc_vec(ref_mfcc_vec, sample_mfcc_vec):
    # Compute cosine similarity
    similarity = cosine_similarity(ref_mfcc_vec, sample_mfcc_vec)
    return similarity[0][0]


In [ ]:
!pip install langchain_core

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
!pip install pydub

# TARGET AUDIO Q & A

In [ ]:
# EM & LF
video_url = "https://www.youtube.com/watch?v=smK9dgdTl40"

In [ ]:

import re
import os
from fastapi import FastAPI, UploadFile, HTTPException
from youtube_transcript_api import YouTubeTranscriptApi
from typing import List, Dict, Optional
from pydantic import BaseModel, HttpUrl
import logging
from tenacity import retry, stop_after_attempt, retry_if_exception_type, wait_fixed
import redis
import json
import uvicorn
import yt_dlp
from datetime import timedelta

video_id = extract_video_id(video_url)

transcript_list = YouTubeTranscriptApi().list(video_id=video_id)

for transcript in transcript_list:
    try:


        if transcript.is_translatable or transcript.is_generated:
            logger.info(
                f"Attempting to fetch transcript {transcript.language_code} for {video_id}"
            )
            transcript = transcript.fetch()
            break
    except Exception as e:
        logger.warning(
            f"Failed to fetch transcript {transcript.language_code} for {video_id}: {str(e)}"
        )
        continue

segments = [
    {
        "text": getattr(entry, "text", ""),
        "start": str(timedelta(seconds=int(getattr(entry, "start", 0)))),
        "duration": getattr(entry, "duration", 0),
        "end": str(timedelta(seconds=int(round((getattr(entry, "start", 0) + getattr(entry, "duration", 0)), 2))))
    }
    for entry in transcript
]

formatted_transcript = " ".join([segment['text'] for segment in segments])

In [ ]:
!pip install python-dotenv

In [ ]:
# src/anubis/utils/model

import logging
logger = logging.getLogger(__name__)


from typing import Optional

from pydantic import BaseModel
import os
from dotenv import load_dotenv
load_dotenv()

""" TODO: Prevent Rate Limiting and Token Limiting Errors and Handle Message Failures """

def init_model(tools=[], 
               tool_choice: str = "auto", 
               response_format = None, 
               image_to_text_model: bool = True):
    
    # context = GlobalContext()
    model_name = os.getenv("MODEL")
    base_url = os.getenv("LLAMA_API_BASE_URL")
    api_key = os.getenv("LLAMA_API_KEY")
    dev = os.getenv("DEV")

    logger.info(f"dev: {dev}")
    logger.info(f"base_url: {base_url}")
    logger.info(f"model_name: {model_name}")

    # if dev == 'TRUE':
    from langchain_openai import ChatOpenAI
    

    if response_format is None:
        model = ChatOpenAI(
                    model = model_name,
                    base_url = base_url,
                    temperature=0.1,
                    top_p=0.1,
                    api_key = api_key,
                ).bind_tools(
                    # method='json_schema', 
                    tools=tools, 
                    tool_choice=tool_choice, # auto: zero or more tools
                    # strict=True, # model output will be guaranteed to match the schema
                    # include_raw=True # model response (JSON e.g.) and the parsed response (Pydantic e.g.) will be returned
                )
    else: 
        model = ChatOpenAI(
            model = model_name,
            base_url = base_url,
            temperature=0.1,
            top_p=0.1,
            api_key = api_key,
        )
        model = model.with_structured_output(schema=response_format)
    # else: 
    #     from langchain_together import ChatTogether
    #     model = ChatTogether(model=model_name, temperature=0.1)
    return model



In [ ]:
from typing import Annotated

In [ ]:
from pydantic import Field

In [ ]:
!pip install langchain-openai

In [ ]:
# class QAResponse(BaseModel):
#     """
#     This is a list of question and answer pairs. For each question in the input text, identify and include the corresponding question. NEVER change any of the text. keep the text exactly as formatted. Identify and correcsponding question and answer pairs. BOTH the question and the answer are within the given input text. NEVER respond with an answer that was created. ONLY identify and return the questions and corresponding answers in the input text.
#     """
#     question: Annotated[str, Field(description="This is the question")]
#     answer: Annotated[str, Field(description="This is the corresponding response to the answer that is found in the input source text. NEVER GENERATE AN ANSWER TO THE QUESTION. ONLY RETURN THE ANSWER IN THE SOURCE DOCUMENT.")]

# class QAListResponse(BaseModel):
#     """
#     This is a list of question and answer pairs. For each question in the input text, identify and include the corresponding question. NEVER change any of the text. keep the text exactly as formatted. Identify and correcsponding question and answer pairs. BOTH the question and the answer are within the given input text. NEVER respond with an answer that was created. ONLY identify and return the questions and corresponding answers in the input text.
#     """
#     responses: Annotated[List[QAResponse], Field(description="This is a list of question and answer pairs. For each question in the input text, identify and include the corresponding question. NEVER change any of the text. keep the text exactly as formatted. Identify and correcsponding question and answer pairs.")]

In [ ]:
# model = init_model(response_format=QAResponse)
# response = model.invoke(input="how are you? ")

In [ ]:
formatted_transcript = " ".join([segment['text'] for segment in segments])

In [ ]:
formatted_transcript

In [ ]:
segments

In [ ]:
import os
import sys
from pydub import AudioSegment

import tqdm

def extract_clips(audio_path, timestamps, output_dir):
    """Extract audio clips from an audio file based on timestamps."""
    print(f"Loading audio from: {audio_path}")
    audio = AudioSegment.from_file(audio_path)
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Derive base name from input file
    base_name = os.path.splitext(os.path.basename(audio_path))[0]
    
    print(f"Extracting {len(timestamps)} clips to: {output_dir}")
    
    for i, seg in tqdm(enumerate(timestamps)):
        start_ms = int(seg['start'] * 1000)
        end_ms = int(seg['end'] * 1000)
        clip = audio[start_ms:end_ms]
        
        out_filename = f"{base_name}_seg{i:04d}_{seg['start']:.1f}-{seg['end']:.1f}.mp3"
        out_path = os.path.join(output_dir, out_filename)
        clip.export(out_path, format="mp3")
        
        if (i + 1) % 50 == 0 or i == len(timestamps) - 1:
            print(f"  Saved {i + 1}/{len(timestamps)} clips...")
    



In [ ]:
extract_clips('./audio/0m3hGZvD-0s_audio.mp3', timestamps=segments, output_dir=f'clips/{video_id}/')

In [ ]:
# def identify_speaker(reference_audio_file_path, target_audio_file_path):
#     """ Accept a reference and target audio file path.
#         Return a Question and answer dataset where the human (question) is the interviewer and the target (answer) is the AIMessage 
#        """
    
#     """ Identify when a person is speaking. """
#     model = load_silero_vad()
#     wav = read_audio(target_audio_file_path)
#     speech_timestamps = get_speech_timestamps(
#         wav, 
#         model,
#         return_seconds=True
#     )

    """ segment the target audio into clips using the speech_timestamps """

    """ compare the reference audio to each clip """

    """ Add a Human or AI message with timestamp metadata of the time, the speaker, and the content is the text """

